# Part 3 of 6: Vision Transformer: Fine-Tuning Strategies

COMP7015 Group Project – Topic 1: Plant Disease Classification. This notebook covers Section 10: three fine-tuning strategies for a pre-trained ViT-B/16 and an augmentation ablation, following Zhouyang's protocol, and the selection of the final ViT on the validation set.

The notebooks run in order: `01_data` → `02_cnn_resnet` → `03_vit` → `04_test_evaluation` → `05_robustness` → `06_discussion`. The shared code lives in the `plantdisease/` package. Finished training runs and predictions are saved under `OUT_DIR/runs/` and reused, so this notebook can be re-run on its own once the earlier ones have run.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score

from plantdisease.config import SEEDS, SHORT_NAMES, out_path
from plantdisease.data import prepare
from plantdisease.evaluate import (
    compare_seeds,
    mean_sd,
    sd,
    seed_scores,
    stable_errors,
    training_cost,
)
from plantdisease.experiments import BASE_VIT, save_selected, vit_configs
from plantdisease.models import build_model, count_params, set_trainable
from plantdisease.plots import finish, mean_and_sd, stack_curves
from plantdisease.train import predict_labels, run_seeds

data = prepare()  # leaf-group split of notebook 01, rebuilt from the image cache

## 10. ViT-B/16 Fine-Tuning (Zhouyang's protocol)

**Model.** torchvision's ViT-B/16 with ImageNet-1K weights. The 224×224 image is cut into 14×14 = 196 patches of 16×16 pixels; together with a class token they pass through 12 self-attention blocks that relate every patch to every other patch (Lab 5). The ImageNet head is replaced by a 4-class linear layer (3,076 parameters); the model has 85.8 M parameters in total.

**Fine-tuning strategies.** All strategies first train **only the new head for 3 epochs**. From epoch 4 they differ only in which parameters are unfrozen:

| Strategy | Epochs 1–3 | Epochs 4–15 |
|---|---|---|
| **head only** | head | head; the backbone stays frozen |
| **partial** | head | last 2 encoder blocks, final LayerNorm and head |
| **full** | head | every parameter, including patch embedding, position embedding and class token |

An **augmentation ablation** repeats the partial strategy without training augmentation. Retraining on grayscale and background-removed photos (Zhouyang's input ablation) follows in notebook 05, Section 15.

**Training settings.** 15 epochs, batch size 16, AdamW with weight decay 0.01, constant learning rates: head 1e-3 in epochs 1–3, then head 1e-4 and unfrozen backbone 1e-5 (two parameter groups; the optimizer is rebuilt at the switch). Cross-entropy without class weights, gradient-norm limit 1, mixed precision on CUDA, the checkpoint with the best validation macro-F1 is kept. **Early stopping** (an addition to Zhouyang's fixed 15 epochs): a run stops when the validation macro-F1 has not improved for 5 epochs of the fine-tuning stage, so strategies can train for different numbers of epochs; the tables report the epochs trained next to the training time. Seeds 62, 63 and 64; a strategy's three runs share the split, and the same seed gives the same initialisation and the same first three epochs for all strategies.

**Differences to Zhouyang's original runs.** The classes, the split and the preprocessing are the project's (notebook 01): the whole photo is resized to 224×224 instead of a 256 resize with a centre crop, and the training augmentation is the project's (flips, 90° rotations, mild brightness/contrast) instead of flips only. The "no augmentation" ablation switches this whole augmentation off.

### 10.1 Which parameters does each strategy train?

A check before training (as in Zhouyang's smoke test): the number of trainable parameters in each stage.

In [2]:
model = build_model(BASE_VIT)
total = count_params(model)
rows = []
for name, cfg in vit_configs.items():
    row = {"run": name}
    for stage in ("head", "finetune"):
        set_trainable(model, cfg, stage)
        row[f"trainable, {stage} stage"] = count_params(model, trainable_only=True)
    row["share of all parameters"] = f"{row['trainable, finetune stage'] / total:.2%}"
    rows.append(row)
del model
pd.DataFrame(rows).set_index("run")

,"trainable, head stage","trainable, finetune stage",share of all parameters
run,,,
ViT-B/16 head only,3076,3076,0.00%
ViT-B/16 partial (last 2 blocks),3076,14180356,16.53%
ViT-B/16 full fine-tuning,3076,85801732,100.00%
"ViT-B/16 partial, no augmentation",3076,14180356,16.53%


### 10.2 Training

Every configuration is trained with all three seeds (4 configurations × 3 seeds = 12 runs).

In [3]:
vit_histories = {name: run_seeds(name, data) for name in vit_configs}


===== ViT-B/16 head only (seed 62) =====
Epoch 1/15 [head] | train loss 0.3315 acc 0.9015 | val loss 0.1514 acc 0.9557 macro-F1 0.9557 | 87s
Epoch 2/15 [head] | train loss 0.1030 acc 0.9737 | val loss 0.0901 acc 0.9729 macro-F1 0.9715 | 83s
Epoch 3/15 [head] | train loss 0.0661 acc 0.9830 | val loss 0.0675 acc 0.9757 macro-F1 0.9747 | 84s
Epoch 4/15 [finetune] | train loss 0.0501 acc 0.9883 | val loss 0.0609 acc 0.9829 macro-F1 0.9823 | 85s
Epoch 5/15 [finetune] | train loss 0.0522 acc 0.9880 | val loss 0.0597 acc 0.9843 macro-F1 0.9837 | 81s
Epoch 6/15 [finetune] | train loss 0.0494 acc 0.9895 | val loss 0.0573 acc 0.9871 macro-F1 0.9867 | 81s
Epoch 7/15 [finetune] | train loss 0.0455 acc 0.9917 | val loss 0.0564 acc 0.9857 macro-F1 0.9854 | 77s
Epoch 8/15 [finetune] | train loss 0.0451 acc 0.9920 | val loss 0.0549 acc 0.9857 macro-F1 0.9850 | 79s
Epoch 9/15 [finetune] | train loss 0.0441 acc 0.9898 | val loss 0.0535 acc 0.9857 macro-F1 0.9850 | 79s
Epoch 10/15 [finetune] | train los

KeyboardInterrupt: 

### 10.3 Validation results, cost and selection

The final ViT is the configuration with the highest **mean validation macro-F1 over the seeds**. The table also shows the training cost: trainable parameters in the fine-tuning stage, epochs trained (early stopping), total training time (all epochs, without validation) and the peak memory during training.

In [ ]:
rows = []
for name, hs in vit_histories.items():
    rows.append(
        {
            "run": name,
            "val macro-F1": mean_sd([max(h["val_f1"]) for h in hs]),
            "best epochs": ", ".join(str(int(np.argmax(h["val_f1"])) + 1) for h in hs),
            **training_cost(name),
        }
    )
vit_validation = pd.DataFrame(rows).set_index("run")
vit_validation.to_csv(out_path("vit_validation.csv"))

mean_val = {
    n: np.mean([max(h["val_f1"]) for h in hs]) for n, hs in vit_histories.items()
}
BEST_VIT = max(mean_val, key=mean_val.get)
# OUT_DIR/final_models.json, read by notebooks 04-06
save_selected("vit_b_16", BEST_VIT)
print("Selected ViT:", BEST_VIT)
vit_validation

In [ ]:
# Learning curves: mean over the seeds, shaded ± sd; dashed line: switch from head-only to fine-tuning
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k, (name, hs) in enumerate(vit_histories.items()):
    for ax, key in zip(axes, ("val_loss", "val_f1"), strict=True):
        values = stack_curves(hs, key)  # NaN after a seed stopped early
        epochs = np.arange(1, values.shape[1] + 1)
        mean, spread = mean_and_sd(values)
        ax.plot(epochs, mean, color=f"C{k}", label=name)
        ax.fill_between(epochs, mean - spread, mean + spread, color=f"C{k}", alpha=0.2)
for ax, label in zip(axes, ("validation loss", "validation macro-F1"), strict=True):
    ax.axvline(
        BASE_VIT["head_epochs"] + 0.5, color="black", linestyle="--", linewidth=0.8
    )
    ax.set_xlabel("epoch")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[1].legend(fontsize=8)
finish("vit_learning_curves.png")

### 10.4 Test results of all strategies (after selection)

The selection above uses only the validation set. Afterwards, every strategy is evaluated once on the test set, as in Zhouyang's comparison, to show what the strategies achieve and cost. Strategies trained with the same seed share the initialisation and the first three epochs, so they are also compared seed by seed. The table of errors counts the test photos that every seed misclassifies and those misclassified at least once.

In [ ]:
y_test = data.y_test
vit_preds = predict_labels(vit_configs, {"test": data.images[data.test_idx]})
test_f1 = seed_scores(vit_configs, vit_preds, "test", y_test)
rows = []
for name in vit_configs:
    preds = [vit_preds[(name, s, "test")] for s in SEEDS]
    rows.append(
        {
            "run": name,
            "test accuracy": mean_sd([(p == y_test).mean() for p in preds]),
            "test macro-F1": mean_sd(test_f1[name]),
            **stable_errors(preds, y_test),
            "train time (s)": training_cost(name)["train time (s)"],
            "peak memory (MiB)": training_cost(name)["peak memory (MiB)"],
        }
    )
vit_test = pd.DataFrame(rows).set_index("run")
vit_test.to_csv(out_path("vit_strategies_test.csv"))
vit_test

In [ ]:
# Same-seed comparisons on the test set (A - B in macro-F1)
names = list(vit_configs)
pairs = {
    "partial vs. head only": (names[1], names[0]),
    "full vs. partial": (names[2], names[1]),
    "augmentation vs. none (partial)": (names[1], names[3]),
}
vit_comparison = pd.DataFrame(
    {label: compare_seeds(test_f1[a], test_f1[b]) for label, (a, b) in pairs.items()}
).T
vit_comparison.to_csv(out_path("vit_strategy_comparison.csv"))
vit_comparison

In [ ]:
# Per-class test F1 (mean over seeds) and the accuracy-cost trade-off of the strategies
per_class = pd.DataFrame(
    {
        name: np.mean(
            [
                f1_score(y_test, vit_preds[(name, s, "test")], average=None)
                for s in SEEDS
            ],
            axis=0,
        )
        for name in vit_configs
    },
    index=SHORT_NAMES,
).T
display(per_class.round(4))

cost = pd.DataFrame([training_cost(n) for n in vit_configs], index=list(vit_configs))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, column in zip(axes, ("train s/epoch", "peak memory (MiB)"), strict=True):
    for k, name in enumerate(vit_configs):
        ax.errorbar(
            cost.at[name, column],
            test_f1[name].mean(),
            yerr=sd(test_f1[name]),
            fmt="o",
            color=f"C{k}",
            label=name,
            capsize=3,
        )
    ax.set_xlabel(column)
    ax.set_ylabel("test macro-F1 (mean ± sd)")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
finish("vit_strategy_cost.png")